In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetB0
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Enable Mixed Precision Training for Speedup
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [2]:
# Define dataset directories
train_dir = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images/train"
test_dir = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images/test"

# Data Preprocessing and Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,           
    rotation_range=20,        
    width_shift_range=0.2,    
    height_shift_range=0.2,   
    shear_range=0.2,          
    zoom_range=0.2,           
    horizontal_flip=True,     
    fill_mode='nearest',      
    validation_split=0.2      
)

# Load Train and Validation Data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [3]:
# Load EfficientNetB0 (Lightweight and Fast)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze layers for feature extraction

# Unfreeze Last 20 Layers for Fine-Tuning
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Add Custom Layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)  # Binary classification

model = Model(inputs=base_model.input, outputs=output)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
# Compile Model with Optimized Settings
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the Model with Optimized Settings
history = model.fit(
    train_generator,
    epochs=5,              # Reduce epochs to fit within 12-hour limit
    validation_data=val_generator
)

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1525s 596ms/step - accuracy: 0.4996 - loss: 0.6984 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1105s 441ms/step - accuracy: 0.5000 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1118s 446ms/step - accuracy: 0.4995 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 4/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1082s 432ms/step - accuracy: 0.5036 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 5/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1130s 451ms/step - accuracy: 0.5020 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6931


In [5]:
# Performance Evaluation
# Load Test Data (No Augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Keep the order for evaluation
)


Found 20000 images belonging to 2 classes.


In [6]:
# Get Predictions
y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob > 0.5).astype(int)  # Convert to binary class

# Get True Labels
y_true = test_generator.classes

# Compute and Print Performance Metrics
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1']))

625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 186ms/step
Confusion Matrix:
[[    0 10000]
 [    0 10000]]

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.00      0.00      0.00     10000
     Class 1       0.50      1.00      0.67     10000

    accuracy                           0.50     20000
   macro avg       0.25      0.50      0.33     20000
weighted avg       0.25      0.50      0.33     20000



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
